# Data Preprocessing and Analysis
This notebook covers the complete data preprocessing workflow for the Rain in Australia dataset. We will load the data, handle missing values, split the dataset, scale numeric features, and encode categorical features before saving the processed files for model training.


### 1. Load Data
We import the necessary libraries (`pandas` and `numpy`), load the raw weather dataset, drop records that lack the target variable (`RainTomorrow`), and inspect the data types and column structures.


In [174]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/row/weatherAUS.csv')
df.dropna(subset=['RainTomorrow'], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 142193 entries, 0 to 145458
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           142193 non-null  object 
 1   Location       142193 non-null  object 
 2   MinTemp        141556 non-null  float64
 3   MaxTemp        141871 non-null  float64
 4   Rainfall       140787 non-null  float64
 5   Evaporation    81350 non-null   float64
 6   Sunshine       74377 non-null   float64
 7   WindGustDir    132863 non-null  object 
 8   WindGustSpeed  132923 non-null  float64
 9   WindDir9am     132180 non-null  object 
 10  WindDir3pm     138415 non-null  object 
 11  WindSpeed9am   140845 non-null  float64
 12  WindSpeed3pm   139563 non-null  float64
 13  Humidity9am    140419 non-null  float64
 14  Humidity3pm    138583 non-null  float64
 15  Pressure9am    128179 non-null  float64
 16  Pressure3pm    128212 non-null  float64
 17  Cloud9am       88536 non-null   fl

### 2. Extract Date Features
We extract the year from the `Date` column to use it for splits.


In [175]:
year = pd.to_datetime(df['Date']).dt.year

### 3. Split into Train, Validation, and Test Sets
We split the data by year to avoid data leakage and simulate real-time forecasting:
- **Train Set**: Data before 2015
- **Validation Set**: Data from 2015
- **Test Set**: Data after 2015


In [176]:
#create train,validation and test sets

train_df = df[year < 2015]
validation_df = df[year == 2015]
test_df = df[year > 2015]

### 4. Check Dataset Shapes
We print the shape of each split to ensure they are partitioned correctly.


In [177]:
print(f"Train set: {train_df.shape}")
print(f"Validation set: {validation_df.shape}")
print(f"Test set: {test_df.shape}")

Train set: (98988, 23)
Validation set: (17231, 23)
Test set: (25974, 23)


### 5. Identify Input and Target Columns
We define the list of input columns (features) and identify the target column we want to predict (`RainTomorrow`).


In [178]:
input_cols = list(df.columns)[1:-1]
target_col = 'RainTomorrow'

### 6. Separate Inputs and Targets
We create copies of the inputs and target variables for training, validation, and test datasets.


In [179]:
train_inputs = train_df[input_cols].copy()
train_targets = train_df[target_col].copy()

validation_inputs = validation_df[input_cols].copy()
validation_targets = validation_df[target_col].copy()

test_inputs = test_df[input_cols].copy()
test_targets = test_df[target_col].copy()

### 7. Identify Numeric Columns
We extract columns with numeric data types to apply imputation and scaling.


In [180]:
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.to_list()

### 8. Identify Categorical Columns
We extract columns with object data types to apply encoding later.


In [181]:
categorical_cols = train_inputs.select_dtypes(include=object).columns.to_list()

### 9. Check for Missing Values in Numeric Columns
Before imputation, we inspect the count of missing values in the numeric features of the training set.


In [182]:
train_inputs[numeric_cols].isnull().sum().sort_values(ascending=False)

Sunshine         40696
Evaporation      37110
Cloud3pm         36766
Cloud9am         35764
Pressure9am       9345
Pressure3pm       9309
WindGustSpeed     6902
Humidity9am       1265
Humidity3pm       1186
WindSpeed3pm      1140
WindSpeed9am      1133
Rainfall          1000
Temp9am            783
Temp3pm            663
MinTemp            434
MaxTemp            198
dtype: int64

### 10. Fit Imputer
We initialize and fit a `SimpleImputer` using the `mean` strategy on the training numeric features.


In [183]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean').fit(train_inputs[numeric_cols])

### 11. Impute Missing Values
We transform the numeric features of the training, validation, and test sets using the fitted imputer.


In [184]:
train_inputs[numeric_cols] = imputer.transform(train_inputs[numeric_cols])
validation_inputs[numeric_cols] = imputer.transform(validation_inputs[numeric_cols])
test_inputs[numeric_cols] = imputer.transform(test_inputs[numeric_cols])

### 12. Confirm Imputation
We check again to verify that there are no missing values remaining in the numeric columns.


In [185]:
train_inputs[numeric_cols].isnull().sum().sort_values(ascending=False)

MinTemp          0
MaxTemp          0
Rainfall         0
Evaporation      0
Sunshine         0
WindGustSpeed    0
WindSpeed9am     0
WindSpeed3pm     0
Humidity9am      0
Humidity3pm      0
Pressure9am      0
Pressure3pm      0
Cloud9am         0
Cloud3pm         0
Temp9am          0
Temp3pm          0
dtype: int64

### 13. Import MinMaxScaler
We import `MinMaxScaler` from `sklearn.preprocessing` to prepare for feature scaling.


In [186]:
from sklearn.preprocessing import MinMaxScaler

### 14. Check Numeric Range Before Scaling
We inspect the minimum and maximum values of validation numeric columns to understand their original range.


In [187]:
validation_inputs.describe().loc[['min', 'max']]

,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm
min,-8.2,-3.2,0.0,0.0,0.0,7.0,0.0,0.0,4.0,0.0,988.1,982.2,0.0,0.0,-6.2,-4.0
max,31.9,45.4,247.2,70.4,14.5,135.0,87.0,74.0,100.0,100.0,1039.3,1037.3,8.0,8.0,37.5,42.8


### 15. Fit and Apply Scaler
We fit the `MinMaxScaler` on the training numeric columns and scale all three datasets to the `[0, 1]` range.


In [188]:
scaler = MinMaxScaler().fit(train_inputs[numeric_cols])

train_inputs[numeric_cols] = scaler.transform(train_inputs[numeric_cols])
validation_inputs[numeric_cols] = scaler.transform(validation_inputs[numeric_cols])
test_inputs[numeric_cols] = scaler.transform(test_inputs[numeric_cols])

### 16. Verify Scaling
We verify that the validation inputs now lie within the expected scaled ranges.


In [189]:
validation_inputs.describe().loc[['min', 'max']]

,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm
min,0.007075,0.017241,0.000000,0.000000,0.000000,0.007752,0.0,0.000000,0.04,0.0,0.125620,0.052805,0.000000,0.000000,-0.006508,0.021484
max,0.952830,0.948276,0.666307,0.854369,1.013986,1.000000,1.0,0.850575,1.00,1.0,0.971901,0.962046,0.888889,0.888889,0.941432,0.935547


### 17. Import OneHotEncoder
We import the `OneHotEncoder` from `sklearn.preprocessing` to encode categorical columns.


In [190]:
from sklearn.preprocessing import OneHotEncoder

### 18. Impute Categorical NaNs
We fill all missing categorical values with the placeholder string `'Unknown'` across all splits.


In [191]:
train_inputs[categorical_cols] = train_inputs[categorical_cols].fillna('Unknown')
validation_inputs[categorical_cols] = validation_inputs[categorical_cols].fillna('Unknown')
test_inputs[categorical_cols] = test_inputs[categorical_cols].fillna('Unknown')

### 19. Fit OneHotEncoder
We fit a `OneHotEncoder` on the training categorical columns and extract the list of new column names.


In [192]:
encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
).fit(train_inputs[categorical_cols])

encoded_cols = encoder.get_feature_names_out(categorical_cols).tolist()

encoded_cols

['Location_Adelaide',
 'Location_Albany',
 'Location_Albury',
 'Location_AliceSprings',
 'Location_BadgerysCreek',
 'Location_Ballarat',
 'Location_Bendigo',
 'Location_Brisbane',
 'Location_Cairns',
 'Location_Canberra',
 'Location_Cobar',
 'Location_CoffsHarbour',
 'Location_Dartmoor',
 'Location_Darwin',
 'Location_GoldCoast',
 'Location_Hobart',
 'Location_Katherine',
 'Location_Launceston',
 'Location_Melbourne',
 'Location_MelbourneAirport',
 'Location_Mildura',
 'Location_Moree',
 'Location_MountGambier',
 'Location_MountGinini',
 'Location_Newcastle',
 'Location_Nhil',
 'Location_NorahHead',
 'Location_NorfolkIsland',
 'Location_Nuriootpa',
 'Location_PearceRAAF',
 'Location_Penrith',
 'Location_Perth',
 'Location_PerthAirport',
 'Location_Portland',
 'Location_Richmond',
 'Location_Sale',
 'Location_SalmonGums',
 'Location_Sydney',
 'Location_SydneyAirport',
 'Location_Townsville',
 'Location_Tuggeranong',
 'Location_Uluru',
 'Location_WaggaWagga',
 'Location_Walpole',
 'Locat

### 20. Encode Categorical Columns
We define a helper function `encode_dataset` that performs the one-hot encoding conversion and combines it with our existing columns.


In [193]:
def encode_dataset(df, encoder, categorical_cols):
    encoded = pd.DataFrame(
        encoder.transform(df[categorical_cols]),
        columns=encoder.get_feature_names_out(categorical_cols),
        index=df.index
    )
    return pd.concat(
        [df.drop(columns=categorical_cols), encoded],
        axis=1
    )

train_inputs = encode_dataset(train_inputs, encoder, categorical_cols)
validation_inputs = encode_dataset(validation_inputs, encoder, categorical_cols)
test_inputs = encode_dataset(test_inputs, encoder, categorical_cols)

### 21. Select Final Columns
We filter our training, validation, and test inputs to include only the numeric and one-hot encoded columns.


In [194]:
train_inputs = train_inputs[numeric_cols + encoded_cols]
validation_inputs = validation_inputs[numeric_cols + encoded_cols]
test_inputs = test_inputs[numeric_cols + encoded_cols]

### 22. Save Processed Datasets
Finally, we save the preprocessed datasets as CSV files to the `../data/processed/` directory for model training.


In [195]:
from pathlib import Path

save_dir = Path("../data/processed")
save_dir.mkdir(parents=True, exist_ok=True)

train_inputs.to_csv(save_dir / "train_inputs.csv", index=False)
train_targets.to_csv(save_dir / "train_targets.csv", index=False)

validation_inputs.to_csv(save_dir / "validation_inputs.csv", index=False)
validation_targets.to_csv(save_dir / "validation_targets.csv", index=False)

test_inputs.to_csv(save_dir / "test_inputs.csv", index=False)
test_targets.to_csv(save_dir / "test_targets.csv", index=False)

print("Datasets saved successfully!")

Datasets saved successfully!
